<a href="https://colab.research.google.com/github/nitshar002/real-estate-pricing-model/blob/main/AdvancedProcessing_CaliforniaRealEstateProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Importing Initial Preprocessing File

In [ ]:
user = 'Nitya'
user = user.lower()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

train_df = pd.read_parquet(
    "/content/drive/MyDrive/IDX Housing Data/initial_train_processed.parquet"
)

test_df = pd.read_parquet(
    "/content/drive/MyDrive/IDX Housing Data/initial_test_processed.parquet"
)


ValueError: mount failed

### Dropping Columns

Comment drops if you want to keep these columns.

After meeting, I think we should keep Lat and Longitude

In [ ]:
# Drop for sure, redundant columns. Flooring is OHE manually, Zip3 is perfectly multicollinear with zipcode, which contains more data.
# Also removed PostalCode_root and PostalCode5, as PostalCode is already adjusted to be a 5 digit zipcode.
train_df = train_df.drop(columns=['Flooring', 'Zip3', 'PostalCode_root', 'PostalCode5', 'UnparsedAddress'])
test_df = test_df.drop(columns=['Flooring', 'Zip3', 'PostalCode_root', 'PostalCode5'])

# Probably drop based on multicolinnearity and bias. PostalCode is related to all of these locational variables.
train_df = train_df.drop(columns=['CountyOrParish', 'City', 'StateOrProvince'])
test_df = test_df.drop(columns=['CountyOrParish', 'City', 'StateOrProvince',])

# Maybe drop. Depends how we want to evaluate YearBuilt and LivingArea
train_df = train_df.drop(columns=['YearBuiltBin', 'LivingAreaBin'])
test_df = test_df.drop(columns=['YearBuiltBin', 'LivingAreaBin'])

train_df = train_df.dropna(subset=['LivingArea'])
test_df = test_df.dropna(subset=['LivingArea'])

# # Probably keep. Use Lat/Long to fill other nulls in HighSchoolDistrict.


# # Then maybe drop Lat/Long afterwards? not sure.
# train_df = train_df.drop(columns=['Latitude', 'Longitude', 'HighSchoolDistrict'])
# test_df = test_df.drop(columns=['Latitude', 'Longitude', 'HighSchoolDistrict'])

# Realistically we should keep one or the other, Lat/Long or PostalCode for multicollinearity purposes.
# But maybe keeping BOTH is strong b/c Location is very, very important for the model.
# Tyler Math Class Note (probably OD):
# Create orthogonal polynomials with the similar vectors to create distinct, independent features and reduce error.
# Probably won't do much as this is a very very small tweak, but I might implement it bc I'm learning this content in school rn.

In [ ]:
# Grab the very first row (index 0) from our 10-row sample
first_house = test_df.iloc[1]

# Convert the single row into a vertical DataFrame so it's easy to read
pd.DataFrame({'Feature Value': first_house})

# High School District Encoding

Adding High School District based on Latitude and Longitude

In [ ]:
import geopandas as gpd

# Analyzing data with prints.
print(test_df['HighSchoolDistrict'].value_counts())
print(test_df['HighSchoolDistrict'].isna().sum())
print('Avg of NA:', train_df[train_df['HighSchoolDistrict'].isna()]['ClosePrice'].mean())
print('Avg non NA:', train_df[train_df['HighSchoolDistrict'].notna()]['ClosePrice'].mean())
print('Avg of NA:', test_df[test_df['HighSchoolDistrict'].isna()]['ClosePrice'].mean())
print('Avg non NA:', test_df[test_df['HighSchoolDistrict'].notna()]['ClosePrice'].mean())

# 2. Convert your DataFrame into a GeoDataFrame (turning Lat/Lon into spatial points)
# Make sure your columns are actually named 'Longitude' and 'Latitude'
gdf_points = gpd.GeoDataFrame(
    train_df,
    geometry=gpd.points_from_xy(train_df['Longitude'], train_df['Latitude']),
    crs="EPSG:4326" # Standard GPS coordinate system
)

# 3. Load the GeoJSON District Map you uploaded
gdf_districts = gpd.read_file('/content/drive/MyDrive/IDX Housing Data/California_School_District_Areas_2024-25.geojson')

# 4. Filter the map for only High School and Unified Districts
high_schools_gdf = gdf_districts[gdf_districts['DistrictType'].isin(['High', 'Unified'])]

# 5. Make sure both maps use the exact same coordinate system
gdf_points = gdf_points.to_crs(high_schools_gdf.crs)

# 6. Perform the Spatial Join! (This matches the point to the polygon it falls inside)
joined_data = gpd.sjoin(gdf_points, high_schools_gdf, how="left", predicate="within")

# 7. Grab the 'DistrictName' from the map and fill your empty column
# I chose to completely override the Districts because there were a lot of mismatches in formatting between original and new data,
# and borders could have been changed from year to year, so it is consistent to stay with the same dataset from the same year.
train_df['HighSchoolDistrict'] = joined_data['DistrictName']
train_df['HighSchoolDistrict'] = train_df['HighSchoolDistrict'].fillna('Out of Bounds / Premium')

# Repeating for the test dataframe
# 2. Convert your DataFrame into a GeoDataFrame (turning Lat/Lon into spatial points)
# Make sure your columns are actually named 'Longitude' and 'Latitude'
gdf_points = gpd.GeoDataFrame(
    test_df,
    geometry=gpd.points_from_xy(test_df['Longitude'], test_df['Latitude']),
    crs="EPSG:4326" # Standard GPS coordinate system
)

# 6. Perform the Spatial Join! (This matches the point to the polygon it falls inside)
joined_data = gpd.sjoin(gdf_points, high_schools_gdf, how="left", predicate="within")

# 7. Grab the 'DistrictName' from the map and fill your empty column
# I chose to completely override the Districts because there were a lot of mismatches in formatting between original and new data,
# and borders could have been changed from year to year, so it is consistent to stay with the same dataset from the same year.
test_df['HighSchoolDistrict'] = joined_data['DistrictName']
test_df['HighSchoolDistrict'] = test_df['HighSchoolDistrict'].fillna('Out of Bounds / Premium')

# Checking with prings
print(test_df.isna().sum())
print('Avg of NA:', test_df[test_df['HighSchoolDistrict'].isna()]['ClosePrice'].mean())
print('Avg non NA:', test_df[test_df['HighSchoolDistrict'].notna()]['ClosePrice'].mean())
print(test_df['HighSchoolDistrict'].value_counts())
# print(test_df['HighSchoolDistrict']['Out of Bounds / Premium'])

Adding Encoded School District, similar to how it was done with postal code:

In [ ]:
# 1. Calculate the average price for each district using ONLY the training data
district_price_map = train_df.groupby('HighSchoolDistrict')['ClosePrice'].mean()

# 2. Map those averages onto both your train and test sets
train_df['District_Avg_Price'] = train_df['HighSchoolDistrict'].map(district_price_map)
test_df['District_Avg_Price'] = test_df['HighSchoolDistrict'].map(district_price_map)

# Deal with "Rare" Districts
# 3. Handle "New" Districts (The Rare Fallback Method)
# Count how often each district appears in Train
district_counts = train_df['HighSchoolDistrict'].value_counts()

# Identify "Rare" Districts (e.g., appeared less than 3 times)
# Note: You can adjust this threshold (3) based on your own EDA/elbow method!

dist_stats = train_df.groupby('PostalCode')['ClosePrice'].agg(['count', 'mean', 'std'])

# Elbow method found 3 again:
# print(dist_stats[dist_stats['count'] == 2]['std'].mean())
# print(dist_stats[dist_stats['count'] == 3]['std'].mean())
# print(dist_stats[dist_stats['count'] == 4]['std'].mean())
# print(dist_stats[dist_stats['count'] == 5]['std'].mean())
# print(dist_stats[dist_stats['count'] == 10]['std'].mean())

rare_districts = district_counts[district_counts < 3].index

# Calculate the average price of these "Rare" districts using ONLY training data
rare_district_mean = train_df[train_df['HighSchoolDistrict'].isin(rare_districts)]['ClosePrice'].mean()

print(f"Global District Mean: ${train_df['ClosePrice'].mean():,.0f}")
print(f"Rare District Mean: ${rare_district_mean:,.0f}")

# 4. Fill the missing Encoded Districts in the test set with the rare district mean
test_df['District_Avg_Price'] = test_df['District_Avg_Price'].fillna(rare_district_mean)

# Now you can drop the original text column!
train_df = train_df.drop(columns=['HighSchoolDistrict'])
test_df = test_df.drop(columns=['HighSchoolDistrict'])

In [ ]:
print(train_df.isna().sum())
print(test_df.isna().sum())

print(train_df.shape)
print(test_df.shape)

# Maybe MainLevelBedrooms is redundant with BedroomsTotal?

### PostalCode Encoding

Uncomment the block(s) you want to test in the model.

CHARLIE Dummies method for PostalCode:

In [ ]:
# # Testing to make sure that ALL dummy columns come from postal code.
# # train_df_test = train_df.copy()
# # train_df_test.drop(columns='PostalCode', inplace=True)
# # train_df_test = pd.get_dummies(train_df_test, drop_first=True)

# train_df_encoded = pd.get_dummies(train_df, drop_first=True)
# test_df_encoded = pd.get_dummies(test_df, drop_first=True)

# train_df_encoded, test_df_encoded = train_df_encoded.align(
#     test_df_encoded,
#     join='left',
#     axis=1,
#     fill_value=0
# )

# # print(train_df_encoded.columns[32:-33])

# print(train_df_encoded.shape)
# # print(train_df_test.shape)

TYLER Encoding PostalCode grouped by average ClosePrice per PostalCode. Also factored in rare ZIPs, and gave them a standardized ClosePrice.

Faster Model compared to dummies, but more likely to overfit.

In [ ]:
# Check if 'PostalCode' column exists before proceeding with encoding
if 'PostalCode' in train_df.columns:
    # 1. Calculate the average price for each Zip Code in the TRAINING set
    zip_code_means = train_df.groupby('PostalCode')['ClosePrice'].mean()

    # 2. Map these averages to the Train set
    train_df['Postal_Code_Encoded'] = train_df['PostalCode'].map(zip_code_means)

    # 3. Map the SAME averages to the Test set
    test_df['Postal_Code_Encoded'] = test_df['PostalCode'].map(zip_code_means)

    # 4. Handle "New" Zip Codes (The Safety Net)
    # If it is new in train and not in test, it is a rare Zipcode, likely a rural area.
    # Count how often each zip appears in Train
    zip_counts = train_df['PostalCode'].value_counts()

    # Identify "Rare" Zips (e.g., appeared less than 3 times, used elbow method to decide 3)
    rare_zips = zip_counts[zip_counts < 3].index

    # Elbow Method
    # print(len(rare_zips)/train_df.shape[0])
    # print(test_df['Postal_Code_Encoded'].isna().sum()/test_df.shape[0])

    # zip_stats = train_df.groupby('PostalCode')['ClosePrice'].agg(['count', 'mean', 'std'])
    # # Look at the standard deviation (volatility) for low counts
    # print(zip_stats[zip_stats['count'] == 2]['std'].mean())
    # print(zip_stats[zip_stats['count'] == 3]['std'].mean())
    # print(zip_stats[zip_stats['count'] == 4]['std'].mean())
    # print(zip_stats[zip_stats['count'] == 5]['std'].mean())
    # print(zip_stats[zip_stats['count'] == 10]['std'].mean())

    # Calculate the average price of these "Rare" zips
    rare_zip_mean = train_df[train_df['PostalCode'].isin(rare_zips)]['ClosePrice'].mean()

    train_data_mean = test_df[test_df['Postal_Code_Encoded'].isna()]['ClosePrice'].mean()

    print(f"Global Mean: ${train_df['ClosePrice'].mean():,.0f}")
    print(f"Rare Zip Mean: ${rare_zip_mean:,.0f}")
    print(f"Train Data Mean: ${train_data_mean:,.0f}")
    # This gives us a more accurate representation of train data,
    # without using the train data itself

    # Fill the Encoded Postal Code with the rare zipcode mean.
    test_df['Postal_Code_Encoded'] = test_df['Postal_Code_Encoded'].fillna(rare_zip_mean)

    # Drop the original 'PostalCode' column after encoding
    train_df.drop(columns=['PostalCode'], inplace=True)
    test_df.drop(columns=['PostalCode'], inplace=True)
else:
    print("Warning: 'PostalCode' column not found in DataFrame. Assuming it has already been encoded and dropped in a previous run.")
    # If 'PostalCode' is not present, assume encoding has already happened.
    # You might want to add a check here to ensure 'Postal_Code_Encoded' column exists if this path is taken.

# Feature Engineering
I will engineer the following features:

*   Log HOA
*   Home age
*   Bed-to-bath ratio
*   Living area to bedrooms
*   Living area per story

I will engineer log HOA to remove the right skewness from HOA costs.

I will engineer home age, so it starts from 0 and will be easier to interpret than just the year the home was built.

A bed-to-bath ratio will directly compare the two features. A ratio closer to 1 would be more ideal.

For living area to bedrooms, perhaps a larger ratio would mean more living space per pereson and we can directly account for this

Lastly, a feature for living area per story will offer insight into the architecture and style of the house.

Location seems to be the most important factor for housing price prediction. We could also factor in restauraunts within 1 km, walk score, distance to downtown. Also distance to coast would be great.

In [ ]:
# Examining weird, negative Monthly_HOA values in our dataframe

print(train_df["Monthly_HOA"].min())
#train_df[train_df["Monthly_HOA"] <= -1]
print((train_df["Monthly_HOA"] == -1).mean())

In [ ]:
# Feature engineering code!

# Replacing the -1 HOA values with 0, then using a log transformation
train_df["Monthly_HOA"] = train_df["Monthly_HOA"].replace(-1, 0)
train_df["log_HOA"] = np.log1p(train_df["Monthly_HOA"])

test_df["Monthly_HOA"] = test_df["Monthly_HOA"].replace(-1, 0)
test_df["log_HOA"] = np.log1p(test_df["Monthly_HOA"])


# This decreased metrics a little, so I'll keep Monthly_HOA and log_HOA
'''
train_df = train_df.drop("Monthly_HOA", axis = 1)
test_df = test_df.drop("Monthly_HOA", axis = 1)
'''
'''
# Engineering PPSF - POTENTIAL LEAKAGE?
train_df["PPSF"] = train_df["ClosePrice"] / train_df["LivingArea"]
test_df["PPSF"] = test_df["ClosePrice"] / test_df["LivingArea"]
'''
# Engineering a home age feature from year built
train_df["Home_Age"] = (2026 - train_df["YearBuilt"])

test_df["Home_Age"] = (2026 - test_df["YearBuilt"])

# Engineering Bed-to-Bath ratio
train_df["Bed_to_Bath"] = np.where(
    train_df["BathroomsTotalInteger"] > 0,
    train_df["BedroomsTotal"] / train_df["BathroomsTotalInteger"],
    np.nan
)

test_df["Bed_to_Bath"] = np.where(
    test_df["BathroomsTotalInteger"] > 0,
    test_df["BedroomsTotal"] / test_df["BathroomsTotalInteger"],
    np.nan
)

median_ratio = train_df["Bed_to_Bath"].median()

train_df["Bed_to_Bath"].fillna(median_ratio, inplace=True)
test_df["Bed_to_Bath"].fillna(median_ratio, inplace=True)


# Engineering living area to bedrooms ratio
train_df["Living_Area_to_Bedrooms"] = np.where(
    train_df["BedroomsTotal"] > 0,
    train_df["LivingArea"] / train_df["BedroomsTotal"],
    np.nan
)

test_df["Living_Area_to_Bedrooms"] = np.where(
    test_df["BedroomsTotal"] > 0,
    test_df["LivingArea"] / test_df["BedroomsTotal"],
    np.nan
)

median_ratio = train_df["Living_Area_to_Bedrooms"].median()

train_df["Living_Area_to_Bedrooms"].fillna(median_ratio, inplace=True)
test_df["Living_Area_to_Bedrooms"].fillna(median_ratio, inplace=True)


# Engineering living area per story
train_df["Living_Area_per_Story"] = train_df["LivingArea"] / train_df["Stories"]

test_df["Living_Area_per_Story"] = test_df["LivingArea"] / test_df["Stories"]

In [ ]:
# Code to avoid errors with pandas 3.0

'''
# Replace HOA -1 values and log transform
train_df["Monthly_HOA"] = train_df["Monthly_HOA"].replace(-1, 0)
train_df["log_HOA"] = np.log1p(train_df["Monthly_HOA"])

test_df["Monthly_HOA"] = test_df["Monthly_HOA"].replace(-1, 0)
test_df["log_HOA"] = np.log1p(test_df["Monthly_HOA"])

train_df.drop("Monthly_HOA", axis=1, inplace=True)
test_df.drop("Monthly_HOA", axis=1, inplace=True)

# Living area to bedroom ratio
train_df["Living_Area_to_Bedrooms"] = np.where(
    train_df["BedroomsTotal"] > 0,
    train_df["LivingArea"] / train_df["BedroomsTotal"],
    np.nan
)

test_df["Living_Area_to_Bedrooms"] = np.where(
    test_df["BedroomsTotal"] > 0,
    test_df["LivingArea"] / test_df["BedroomsTotal"],
    np.nan
)

median_ratio = train_df["Living_Area_to_Bedrooms"].median()

train_df["Living_Area_to_Bedrooms"] = train_df["Living_Area_to_Bedrooms"].fillna(median_ratio)
test_df["Living_Area_to_Bedrooms"] = test_df["Living_Area_to_Bedrooms"].fillna(median_ratio)

# Living area per story
train_df["Living_Area_per_Story"] = train_df["LivingArea"] / train_df["Stories"]
test_df["Living_Area_per_Story"] = test_df["LivingArea"] / test_df["Stories"]
'''



In [ ]:
train_df[["Latitude","Longitude"]].isna().sum()

In [ ]:
# Collecting the restauraunt data. It is saved as a parquet, so no need to run this block.

'''
import osmnx as ox
ox.settings.use_cache = True

# Increase the max query area size to prevent issues with large bounding boxes
# The default is 250_000_000. The warning suggested 222 times this.
# Setting it to a very large number to try to prevent subdivision issues.
ox.settings.max_query_area_size = 60_000_000_000 # ~240 times the default

# The previous approach of using features_from_bbox with a custom, buffered bounding box
# for an entire state often leads to complex geometries that cause errors.
# Using features_from_place for a well-defined administrative boundary is more robust.

# Use features_from_place for better handling of large administrative areas
restaurants = ox.features_from_place(
    "California, USA",
    tags={"amenity": "restaurant"}
)

restaurants = restaurants[restaurants.geometry.type == "Point"]

restaurants["lat"] = restaurants.geometry.y
restaurants["lon"] = restaurants.geometry.x

# Save to Google Drive to ensure persistence
restaurants_path = f"/content/drive/MyDrive/IDX Housing Data/{user}_restaurants.parquet"
restaurants[["lat","lon"]].to_parquet(restaurants_path)
'''

In [ ]:
# Restauraunt feature engineering

restaurants = pd.read_parquet(
    f"/content/drive/MyDrive/IDX Housing Data/zengtao_restaurants.parquet"
)


# Houses from train_df
houses_train = train_df[["Latitude","Longitude"]]
houses_test = test_df[["Latitude","Longitude"]]

# Convert coordinates to radians so we can find distance
coords_houses_train = np.radians(train_df[["Latitude","Longitude"]].values)
coords_houses_test = np.radians(test_df[["Latitude","Longitude"]].values)
coords_rest = np.radians(restaurants[["lat","lon"]].values)

# Build spatial index
from sklearn.neighbors import BallTree
tree = BallTree(coords_rest, metric="haversine")

# Query nearest restaurant for TRAIN
dist_train, idx_train = tree.query(coords_houses_train, k=1)

# Convert radians -> miles
train_df["DistNearestRestaurantMi"] = dist_train.flatten() * 3958.8


# Query nearest restaurant for TEST
dist_test, idx_test = tree.query(coords_houses_test, k=1)

# Convert radians -> miles
test_df["DistNearestRestaurantMi"] = dist_test.flatten() * 3958.8

In [ ]:
train_df.columns

NameError: name 'train_df' is not defined

# Causal Exploration of Features

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

edges = [
    ("LotSizeSquareFeet", "LivingArea"),
    ("LivingArea", "ClosePrice"),
    ("YearBuilt", "LivingArea"),
    ("ClosePrice", "PostalCodeEncoded"),
    ("PostalCodeEncoded", "ClosePrice"), # This is a strong bidirectional relationship. We could engineer more features related to location
    ("YearBuilt", "ClosePrice"),
    ("LivingArea", "BedroomsTotal"),
    ("LivingArea", "ClosePrice"),
    ("BedroomsTotal", "ClosePrice"),
    ("ParkingTotal", "ClosePrice"),
    ("DaysOnMarket", "ClosePrice"),
    ("ClosePrice", "DaysOnMarket"), #But we don't know DaysOnMarket in a realistic prediction scenario. Potential target leakage?
    ("DistNearestRestaurantMi", "ClosePrice"),
    ("DistNearestRestaurantMi", "ClosePrice")
]

G = nx.DiGraph()
G.add_edges_from(edges)

plt.figure(figsize=(8,6))
pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, node_size=3000, node_color="lightblue", font_size=10)
plt.show()

In [ ]:
train_df.columns

### Exporting Train and Test DF to go into Modeling

In [ ]:
print(train_df.isna().sum())
print(test_df.isna().sum())

print(train_df.shape)
print(test_df.shape)

In [ ]:
# user defined in the first box of the file.
train_path = f"/content/drive/MyDrive/IDX Housing Data/{user}_AdvancedProcessingAndModeling/{user}_train_processed.parquet"
test_path = f"/content/drive/MyDrive/IDX Housing Data/{user}_AdvancedProcessingAndModeling/{user}_test_processed.parquet"

train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)